Health Insurance Claim Data Analysis

Dataset Loading

In [ ]:
from google.cloud import bigquery
import pandas as pd
from google.colab import auth

auth.authenticate_user()

client = bigquery.Client(project='project-ef2a3aed-6b63-4885-9aa')

query = """
SELECT *
FROM `project-ef2a3aed-6b63-4885-9aa.Health_Insurance_Data.Healthinsurance`
"""

df = client.query(query).to_dataframe()
df.head()

In [ ]:
df.info()
df.isnull().sum()

In [ ]:
df.head(10)

In [ ]:
data = df.copy()
data = data.dropna(subset=['claim'])
numeric_cols = data.select_dtypes(include=['int64', 'float64', 'Int64', 'Float64']).columns

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].median())

categorical_cols = data.select_dtypes(include=['object', 'string']).columns

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

data.isnull().sum()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import numpy as np
import pandas as pd

y = data['claim']

X = data.drop(columns=['claim'])

categorical_features = X.select_dtypes(include=['object', 'string']).columns.tolist()
numeric_features = X.select_dtypes(include=['int64', 'float64', 'Int64', 'Float64']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append([name, mae, rmse, r2])

results_df = pd.DataFrame(results, columns=['Model', 'MAE', 'RMSE', 'R2'])
results_df

In [ ]:
import pandas as pd
import statsmodels.api as sm

model_data = data.copy()
dependent_variable = 'claim'

independent_variables = [
    'age',
    'sex',
    'weight',
    'bmi',
    'hereditary_diseases',
    'no_of_dependents',
    'smoker',
    'bloodpressure',
    'diabetes'
]


model_data = model_data[[dependent_variable] + independent_variables]

X = model_data[independent_variables]
X = pd.get_dummies(X, drop_first=True)

X = X.astype(float)

y = pd.to_numeric(model_data[dependent_variable], errors='coerce')
X = sm.add_constant(X)
ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)

encoded_cat_features = rf_pipeline.named_steps['preprocessor'] \
    .named_transformers_['cat'] \
    .get_feature_names_out(categorical_features)

all_features = list(encoded_cat_features) + numeric_features

importances = rf_pipeline.named_steps['model'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

feature_importance_df.head(15)

In [ ]:
import matplotlib.pyplot as plt

top_features = feature_importance_df.head(10)

plt.figure(figsize=(10, 6))
plt.barh(top_features['Feature'], top_features['Importance'])
plt.gca().invert_yaxis()
plt.xlabel('Importance')
plt.title('Top Factors Influencing Health Insurance Claims')
plt.show()

In [ ]:
top_features.to_csv("feature_importance.csv", index=False)

print(top_features)